Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-smote_default-model'
add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
n_trials = 50

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=True,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=best_params,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )
    
    
def optimize_smote_grid(
    target:str,
    estimator:object,
    estimator_params:dict,
    param_grid:dict=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=True,
):
    """
    Optimize the SMOTE parameters with GridSearchOptimizer for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics
    )
    
    if param_grid is None:
        param_grid = {
        'sampling_strategy': [0.6, 0.7, 0.8, 0.9, 1.0, 'auto'],
        'k_neighbors': list(range(2,11)),
    }
    opt = GridSearchOptimizer(
        objective=partial(pure_smote_objective, ml_pipe=ml_pipe),
        param_grid=param_grid,
        verbose=True,
    )

    opt.optimize()
    best_params = opt.best_params

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=best_params,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    n_trials=n_trials,
    save_model_and_metrics=False,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-13 13:51:52,892] A new study created in memory with name: smote_study
[I 2025-04-13 13:51:53,058] Trial 0 finished with value: 0.7795918367346939 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.7795918367346939.
[I 2025-04-13 13:51:53,218] Trial 1 finished with value: 0.7826336975273146 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.7826336975273146.
[I 2025-04-13 13:51:53,390] Trial 2 finished with value: 0.800925925925926 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 2 with value: 0.800925925925926.
[I 2025-04-13 13:51:53,551] Trial 3 finished with value: 0.7826336975273146 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 2 with value: 0.800925925925926.
[I 2025-04-13 13:51:53,710] Trial 4 finished with value: 0.8055555555555556 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 4 with value: 0.8055555555

best_params


...

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.883488
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


In [7]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:09,  5.68it/s]

Progress: 1/54.	Score: 0.8055555555555556.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.883488
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


Model saved in ..\results\models_modelling4\LogisticRegression_splashing_smote_opt-smote_default-model


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:07,  7.55it/s]

Progress: 1/54.	Score: 0.8392857142857143.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.785539
holdout_test_accuracy_balanced,0.825
holdout_test_roc_auc,0.955455
holdout_test_f1,0.708333
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.946886


Model saved in ..\results\models_modelling4\LogisticRegression_no_fragmentation_smote_opt-smote_default-model


# KNClassifier

In [9]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:11,  4.61it/s]

Progress: 1/54.	Score: 0.7857142857142857.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_opt-smote...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.874228
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.835913
cv_test_accuracy_balanced_median,0.844427
cv_test_roc_auc_median,0.918731


Model saved in ..\results\models_modelling4\KNeighborsClassifier_splashing_smote_opt-smote_default-model


In [10]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:10,  5.02it/s]

Progress: 1/54.	Score: 0.8250000000000001.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.780518
holdout_test_accuracy_balanced,0.809091
holdout_test_roc_auc,0.937273
holdout_test_f1,0.695652
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.928571


Model saved in ..\results\models_modelling4\KNeighborsClassifier_no_fragmentation_smote_opt-smote_default-model


# SVC

In [11]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:11,  4.44it/s]

Progress: 1/54.	Score: 0.8253119429590017.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_opt-smote_default-model
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.893519
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.863049
cv_test_accuracy_balanced_median,0.885449
cv_test_roc_auc_median,0.922601


Model saved in ..\results\models_modelling4\SVC_splashing_smote_opt-smote_default-model


In [12]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:09,  5.33it/s]

Progress: 1/54.	Score: 0.8346153846153845.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_opt-smote_default-m...
holdout_test_f1_macro,0.857143
holdout_test_accuracy_balanced,0.886364
holdout_test_roc_auc,0.95
holdout_test_f1,0.8
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.877289
cv_test_roc_auc_median,0.948718


Model saved in ..\results\models_modelling4\SVC_no_fragmentation_smote_opt-smote_default-model


# CatBoost

In [13]:
estimator = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:09<08:30,  9.63s/it]

Progress: 1/54.	Score: 0.8940886699507389.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.846814
holdout_test_accuracy_balanced,0.831019
holdout_test_roc_auc,0.959877
holdout_test_f1,0.901961
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.950464


Model saved in ..\results\models_modelling4\CatBoostClassifier_splashing_smote_opt-smote_default-model


In [14]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:10<08:55, 10.10s/it]

Progress: 1/54.	Score: 0.8412698412698413.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.933862
holdout_test_accuracy_balanced,0.947727
holdout_test_roc_auc,0.987273
holdout_test_f1,0.904762
holdout_test_accuracy,0.946667
cv_test_f1_macro_median,0.925457
cv_test_accuracy_balanced_median,0.915751
cv_test_roc_auc_median,0.977778


Model saved in ..\results\models_modelling4\CatBoostClassifier_no_fragmentation_smote_opt-smote_default-model


# XGBoost

In [15]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:18,  2.88it/s]

Progress: 1/54.	Score: 0.8540723981900453.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smote_opt-smote_defaul...
holdout_test_f1_macro,0.852826
holdout_test_accuracy_balanced,0.847222
holdout_test_roc_auc,0.947145
holdout_test_f1,0.897959
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.856037
cv_test_roc_auc_median,0.946032


Model saved in ..\results\models_modelling4\XGBClassifier_splashing_smote_opt-smote_default-model


In [16]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:22,  2.40it/s]

Progress: 1/54.	Score: 0.81524926686217.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smote_opt-smote...
holdout_test_f1_macro,0.882524
holdout_test_accuracy_balanced,0.888636
holdout_test_roc_auc,0.981818
holdout_test_f1,0.829268
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.976068


Model saved in ..\results\models_modelling4\XGBClassifier_no_fragmentation_smote_opt-smote_default-model


# AdaBoost

In [17]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:32,  1.63it/s]

Progress: 1/54.	Score: 0.8392857142857142.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.751994
holdout_test_accuracy_balanced,0.75
holdout_test_roc_auc,0.854552
holdout_test_f1,0.824742
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.794892
cv_test_accuracy_balanced_median,0.794892
cv_test_roc_auc_median,0.891641


Model saved in ..\results\models_modelling4\AdaBoostClassifier_splashing_smote_opt-smote_default-model


In [18]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:30,  1.72it/s]

Progress: 1/54.	Score: 0.8411330049261083.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.853293
holdout_test_accuracy_balanced,0.870455
holdout_test_roc_auc,0.949091
holdout_test_f1,0.790698
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.854396
cv_test_roc_auc_median,0.925824


Model saved in ..\results\models_modelling4\AdaBoostClassifier_no_fragmentation_smote_opt-smote_default-model


# Random Forest

In [19]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:51,  1.03it/s]

Progress: 1/54.	Score: 0.8928571428571428.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smote_opt-smo...
holdout_test_f1_macro,0.8333
holdout_test_accuracy_balanced,0.820602
holdout_test_roc_auc,0.956404
holdout_test_f1,0.891089
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.948413


Model saved in ..\results\models_modelling4\RandomForestClassifier_splashing_smote_opt-smote_default-model


In [20]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:51,  1.02it/s]

Progress: 1/54.	Score: 0.8250000000000001.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smote_...
holdout_test_f1_macro,0.897727
holdout_test_accuracy_balanced,0.897727
holdout_test_roc_auc,0.980455
holdout_test_f1,0.85
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.946154


Model saved in ..\results\models_modelling4\RandomForestClassifier_no_fragmentation_smote_opt-smote_default-model


# LightGBM

In [21]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:23,  2.23it/s]

Progress: 1/54.	Score: 0.8699334543254689.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smote_opt-smote_defau...
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.951775
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.95873


Model saved in ..\results\models_modelling4\LGBMClassifier_splashing_smote_opt-smote_default-model


In [22]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:13,  3.98it/s]

Progress: 1/54.	Score: 0.8503207412687099.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smote_opt-smot...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.980909
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.958974


Model saved in ..\results\models_modelling4\LGBMClassifier_no_fragmentation_smote_opt-smote_default-model


# For the notebook with Model optimization!

In [23]:
# def update_estimator_params(
#     ml_pipe:MLPipeline,
#     suggested_params:dict,
# ) -> dict:
#     """Update the estimator parameters based on the pipeline parameters.

#     Args:
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.
#         suggested_params: A dictionary containing the suggested Estimator hyperparameters.
    
#     Returns:
#         A dictionary containing the estimator parameters.
#     """
#     estimator_params = ml_pipe._pipeline_params['estimator_params']
#     estimator_params.update(suggested_params)
#     return estimator_params

# def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
#     """Objective function for logistic regression optimization using Optuna.

#     Args:
#         trial: An Optuna trial object used to suggest hyperparameters.
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.

#     Returns:
#         The score of the model based on the specified evaluation metric.
#     """
    
#     categorical_features = ('wettability', 'inclination')
    
#     random_state = ml_pipe._pipeline_params['random_state']
    
#     # sample params
#     C = trial.suggest_loguniform('C', 1e-4, 1e3)
    
#     # SMOTE params
#     add_smote = trial.suggest_categorical('add_smote', [True, False])
#     if add_smote:
#         is_smotenc = trial.suggest_categorical('is_smotenc', [True, False])
#         smote_params = {
#             'sampling_strategy': trial.suggest_categorical(
#                 'sampling_strategy', [0.6, 0.7, 0.8, 0.9, 1.0]
#             ),
#             'k_neighbors': trial.suggest_int('k_neighbors', 3, 10),
#             'random_state': random_state,
#         }
#     else:
#         is_smotenc = False
#         smote_params = None
#     if is_smotenc:
#         wettability_cat = trial.suggest_categorical('wettability_cat', [True, False])
#         if wettability_cat:
#             inclination_cat = trial.suggest_categorical('inclination_cat', [True, False])
#         else:
#             inclination_cat = True
        
        
#         mask = [wettability_cat, inclination_cat]
        
#         masked_features = [
#             feature for feature, mask_value in zip(categorical_features, mask) 
#             if mask_value
#         ]
#         smote_params = {
#             **smote_params,
#             'categorical_features': masked_features,
#         }

#     suggested_params = {
#         "C": C,
#     }
    
#     estimator_params = update_estimator_params(ml_pipe, suggested_params)

#     estimator = LogisticRegression(
#         **estimator_params,
#         random_state=random_state,
#     )

#     score = ml_pipe.step(
#         estimator=estimator,
#         add_smote=add_smote,
#         is_smotenc=is_smotenc,
#         smote_params=smote_params,
#     )
    
#     return score



# opt = OptunaOptimizer(
#     objective=partial(logreg_objective, ml_pipe=ml_pipe),
#     study_name="logreg_study",
#     direction="maximize",
# )

# opt.optimize(n_trials=200)

# best_params = opt.study.best_trial.params
# display(best_params)
# # estimator_params = update_estimator_params(ml_pipe, best_params)

# # estimator = LogisticRegression(
# #     **estimator_params,
# #     random_state=ml_pipe._pipeline_params['random_state'],
# # )